In [ ]:
import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)
from kafi.streams.topologynode import TopologyNode as Tn

import cloudpickle as pickle

#

order_source_str = "orders"
order_source_tn = Tn.source(order_source_str)

feedback_source_str = "feedback"
feedback_source_tn = Tn.source(feedback_source_str)

input_with_window_end_tn = (
    order_source_tn
    .map(lambda x: x | {"window_end": x["value"]["ts"] + 10})
)

combined_input_with_window_end_tn = (
    input_with_window_end_tn
    .add(feedback_source_tn)
)

input_cur_ts_tn = (
    combined_input_with_window_end_tn
    .map(lambda x: {"ts": x["value"]["ts"]})
    .max(lambda x: x["ts"],
         lambda x: {"cur_ts": x})
)

join_window_end_tn = (
    combined_input_with_window_end_tn
    .join(input_cur_ts_tn,
          lambda l, r: r["cur_ts"] > l["window_end"],
          lambda l, _: l)
    ._filter(lambda _, w: w > 0)
    .neg()
    ._delay()
)
root_tn = (
    join_window_end_tn
    .add(input_with_window_end_tn)
    ._peek("root")
)
root_tn.build()
print(root_tn.mermaid())
order_source_tn.to_zSet(Tn._from_records)
feedback_source_tn.to_zSet(Tn._from_records)
join_window_end_tn.from_zSet(Tn._to_records)
root_tn.from_zSet(Tn._to_records)
#
def gen_order(order_id_int, ts_int):
    return {"value": {"order_id": order_id_int, "ts": ts_int}}
#
print("Step 1")
root_tn.push({order_source_str: [(gen_order(1, 0), 1)], feedback_source_str: join_window_end_tn.latest()})
root_tn.latest()
print(len(pickle.dumps(root_tn)))
print("Step 2")
root_tn.push({order_source_str: [(gen_order(2, 11), 1)], feedback_source_str: join_window_end_tn.latest()})
root_tn.latest()
print(len(pickle.dumps(root_tn)))
print("Step 3")
root_tn.push({order_source_str: [(gen_order(3, 0), 1)], feedback_source_str: join_window_end_tn.latest()})
root_tn.latest()
print(len(pickle.dumps(root_tn)))
root_tn.push({order_source_str: [], feedback_source_str: join_window_end_tn.latest()})
root_tn.latest()

for i in range(1000):
    print(i)
    root_tn.push({order_source_str: [(gen_order(i + 4, i * 10 + 11), 1)], feedback_source_str: join_window_end_tn.latest()})
    root_tn.latest()
    print(len(pickle.dumps(root_tn)))


In [ ]:
import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)
from kafi.streams.topologynode import TopologyNode as Tn

source1_str = "source_1"
source2_str = "source_2"

source1_tn = Tn.source(source1_str)
source1_tn.to_zSet(Tn._from_records)
source2_tn = Tn.source(source2_str)
source2_tn.to_zSet(Tn._from_records)

root_tn = source1_tn.add(source2_tn)._peek()
root_tn.from_zSet(Tn._to_records)

root_tn.build()

root_tn.push(source1_str, [("bla", 1)])
root_tn.latest()
root_tn.push(source2_str, [("bla", -1)])
root_tn.latest()

In [18]:
import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)
from kafi.streams.topologynode import TopologyNode as Tn

import cloudpickle as pickle

#

order_source_str = "orders"
order_source_tn = Tn.source(order_source_str)
order_source_tn.to_zSet(Tn._from_records)

root_tn = order_source_tn.ttl(
     lambda x: x["value"]["ts"],
     lambda x: x["value"]["ts"] + 10
)._peek()
root_tn.from_zSet(Tn._to_records)

root_tn.build()

#
def gen_order(order_id_int, ts_int):
    return {"value": {"order_id": order_id_int, "ts": ts_int}}
#
print("Step 1")
root_tn.push(order_source_str, [(gen_order(1, 0), 1)])
root_tn.latest()
print(len(pickle.dumps(root_tn)))
print("Step 2")
root_tn.push(order_source_str, [(gen_order(2, 11), 1)])
root_tn.latest()
print(len(pickle.dumps(root_tn)))
print("Step 3")
root_tn.push(order_source_str, [(gen_order(3, 0), 1)])
root_tn.latest()
print(len(pickle.dumps(root_tn)))
root_tn.push(order_source_str, [])
root_tn.latest()

for i in range(1000):
    print(i)
    root_tn.push({order_source_str: [(gen_order(i + 4, i * 10 + 11), 1)], feedback_source_str: join_window_end_tn.latest()})
    root_tn.latest()
    print(len(pickle.dumps(root_tn)))


Step 1
({'value': {'order_id': 1, 'ts': 0}, '_ttl': {'_expiry': 10}}, 1)
25125
Step 2
({'value': {'order_id': 2, 'ts': 11}, '_ttl': {'_expiry': 21}}, 1)
26074
Step 3
({'value': {'order_id': 1, 'ts': 0}, '_ttl': {'_expiry': 10}}, -1)
({'value': {'order_id': 3, 'ts': 0}, '_ttl': {'_expiry': 10}}, 1)
26989
({'value': {'order_id': 3, 'ts': 0}, '_ttl': {'_expiry': 10}}, -1)
0
({'value': {'order_id': 4, 'ts': 11}, '_ttl': {'_expiry': 21}}, 1)
27712
1
({'value': {'order_id': 5, 'ts': 21}, '_ttl': {'_expiry': 31}}, 1)
27775
2
({'value': {'order_id': 6, 'ts': 31}, '_ttl': {'_expiry': 41}}, 1)
27808
3
({'value': {'order_id': 4, 'ts': 11}, '_ttl': {'_expiry': 21}}, -1)
({'value': {'order_id': 2, 'ts': 11}, '_ttl': {'_expiry': 21}}, -1)
({'value': {'order_id': 7, 'ts': 41}, '_ttl': {'_expiry': 51}}, 1)
28041
4
({'value': {'order_id': 5, 'ts': 21}, '_ttl': {'_expiry': 31}}, -1)
({'value': {'order_id': 8, 'ts': 51}, '_ttl': {'_expiry': 61}}, 1)
28218
5
({'value': {'order_id': 6, 'ts': 31}, '_ttl': {